# AnyText Document Generation Feasibility Study (Colab GPU)

**Mục tiêu**: Đánh giá thực nghiệm editor chuyên biệt văn bản **AnyText** (ICLR 2024 Spotlight) trên đúng 2 nguồn thay thế chữ/số (Attempt A & B) từ pilot SDXL đã đóng.
- **Checkpoint**: `damo/cv_anytext_text_generation_editing` (ModelScope v1.1.3, ~6.93 GiB FP16; tắt translator để bảo toàn ngân sách 12 GiB).
- **Dữ liệu**: Tái sử dụng đúng 2 nguồn CORD-v2 validation (Attempt A: row 22, Attempt B: row 61).
- **Bảo toàn hình học**: Native dimensions, context crop 512x512, strict composite inside target bbox, kiểm tra bitwise outside-mask equality.
- **Yêu cầu môi trường**: Google Colab GPU (Tesla T4 miễn phí, >= 15 GB disk, zero-token, zero-login).

In [ ]:
# Cell 1: Hardware Preflight & Disk Inspection
import torch, shutil, os

assert torch.cuda.is_available(), 'ERROR: GPU runtime required! Please enable GPU via Runtime -> Change runtime type.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
disk_free_gb = round(shutil.disk_usage('/').free / (1024**3), 2)

print(f'[PASS] GPU Detected: {gpu_name} (VRAM: {vram_gb} GB)')
print(f'[PASS] Disk Free: {disk_free_gb} GB (Requires >= 15 GB for ModelScope download)')
assert vram_gb >= 7.0, f'VRAM {vram_gb} GB is insufficient for AnyText (requires >= 7 GB).'
assert disk_free_gb >= 12.0, f'Disk space {disk_free_gb} GB is insufficient for 6.93 GiB checkpoint.'

In [ ]:
# Cell 2: Install AnyText Dependencies & Clone Official Repository
import os, sys, subprocess

# Install system font and dependencies
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-dejavu-core'], check=False)
subprocess.run([
    'pip', 'install', '-q',
    'modelscope==1.10.0', 'einops==0.4.1', 'omegaconf==2.2.3',
    'pytorch-lightning==1.5.0', 'open_clip_torch==2.7.0',
    'transformers==4.30.2', 'addict', 'yapf', 'timm'
], check=True)

# Clone AnyText official repository
if not os.path.isdir('AnyText'):
    subprocess.run(['git', 'clone', 'https://github.com/tyxsspa/AnyText.git'], check=True)

os.chdir('AnyText')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('[PASS] Dependencies and AnyText codebase installed.')

In [ ]:
# Cell 3: Staging Geometry Modules, Runner & Plan
import os, pathlib, json

os.makedirs('ml/evaluation', exist_ok=True)
os.makedirs('research/evidence/document_pilot', exist_ok=True)
os.makedirs('inputs/attempt_A_replacement_price', exist_ok=True)
os.makedirs('inputs/attempt_B_replacement_item_name', exist_ok=True)

pathlib.Path('research/evidence/document_pilot/anytext_feasibility_plan.json').write_text("{\n  \"plan_id\": \"anytext_feasibility_plan_v1.0.0\",\n  \"protocol_version\": \"1.3.0\",\n  \"created_at\": \"2026-10-13T20:00:00+07:00\",\n  \"official_project_title\": \"Nghi\u00ean c\u1ee9u v\u00e0 x\u00e2y d\u1ef1ng c\u00f4ng c\u1ee5 ph\u00e1t hi\u1ec7n \u1ea3nh v\u00e0 t\u00e0i li\u1ec7u gi\u1ea3 m\u1ea1o sinh b\u1edfi AI.\",\n  \"objective\": \"Evaluate specialized text editor AnyText on two replacement sources from the closed SDXL pilot to determine feasibility of generating forensically relevant text-edited document samples without full dataset downloads or training.\",\n  \"model_specification\": {\n    \"model_name\": \"AnyText: Multilingual Visual Text Generation And Editing\",\n    \"paper_url\": \"https://arxiv.org/abs/2311.03054\",\n    \"official_repo_url\": \"https://github.com/tyxsspa/AnyText\",\n    \"code_commit_sha\": \"3d4a260c67d92c0e27d6e18ed69990e19a40de6d\",\n    \"modelscope_repo_id\": \"damo/cv_anytext_text_generation_editing\",\n    \"model_revision\": \"v1.1.3\",\n    \"checkpoint_file\": \"anytext_v1.1.ckpt\",\n    \"checkpoint_sha256\": \"09c88d145aa08f067641dc1d10c8890855939fa756d4cdc991e781a5f3918f59\",\n    \"checkpoint_size_bytes\": 5730281759,\n    \"clip_subfolder\": \"clip-vit-large-patch14\",\n    \"clip_weights_size_bytes\": 1710671599,\n    \"total_required_weights_bytes\": 7440953358,\n    \"total_required_weights_gib\": 6.9299,\n    \"translator_subfolder_excluded\": \"nlp_csanmt_translation_zh2en (7,881,046,036 bytes / 7.34 GiB excluded to respect 12 GiB ceiling)\",\n    \"translator_policy\": \"DISABLED (use_translator=False, --no_translator)\",\n    \"dtype\": \"fp16\",\n    \"licenses\": {\n      \"code_license\": \"Apache License 2.0\",\n      \"modelscope_model_license\": \"Apache License 2.0\",\n      \"font_license\": \"Bitstream Vera / Free Software License (SIL OFL compatible)\"\n    },\n    \"font_specification\": {\n      \"font_name\": \"DejaVuSans.ttf\",\n      \"font_package\": \"fonts-dejavu-core\",\n      \"font_path_default_linux\": \"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\",\n      \"font_path_fallback_linux\": \"/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf\",\n      \"font_size_pt\": 60,\n      \"font_source\": \"Debian/Ubuntu system package (open source, verified free distribution)\"\n    }\n  },\n  \"resource_budget\": {\n    \"max_colab_download_gib\": 12.0,\n    \"planned_colab_download_gib\": 6.93,\n    \"max_local_transfer_mib\": 100.0,\n    \"max_gpu_execution_minutes\": 60.0,\n    \"target_gpu\": \"Tesla T4 (15 GB VRAM, Google Colab free tier)\",\n    \"authentication_required\": false\n  },\n  \"geometry_contract\": {\n    \"native_document_dimensions_preserved\": true,\n    \"intermediate_canvas_resolution\": [\n      512,\n      512\n    ],\n    \"context_margin_ratio\": 0.5,\n    \"interpolation_image\": \"BICUBIC\",\n    \"interpolation_mask\": \"NEAREST (strictly binary {0, 255})\",\n    \"outside_mask_equality_contract\": \"np.array_equal == True, max_outside_diff == 0\",\n    \"resolution_difference_vs_sdxl\": \"AnyText uses SD1.5 latent resolution (512x512 canvas) whereas SDXL pilot used 1024x1024. This is a known editor/configuration difference, not an isolated architecture study.\",\n    \"post_generation_policy\": \"Raw model output and final composited document are saved separately. No post-generation typesetting or text stamping is performed.\"\n  },\n  \"attempts\": [\n    {\n      \"attempt_id\": \"attempt_A_replacement_price\",\n      \"source_row_index\": 22,\n      \"source_image_id\": 22,\n      \"operation\": \"replacement\",\n      \"original_text\": \"280,000\",\n      \"target_text\": \"350,000\",\n      \"target_bbox\": [\n        442,\n        406,\n        468,\n        488\n      ],\n      \"native_dimensions\": [\n        576,\n        864\n      ],\n      \"authentic_sha256\": \"b335696aaeb1ba1a607e87cd7acf76798dbc65a188b2f38e347473a762aba087\",\n      \"seed\": 20261012,\n      \"prompt\": \"A receipt bill with price \\\"350,000\\\"\",\n      \"a_prompt\": \"best quality, extremely detailed, clear text edges, neat writing, no watermarks\",\n      \"n_prompt\": \"low-res, bad anatomy, cropped, worst quality, low quality, watermark, unreadable text, messy words, distorted text\",\n      \"sampling_parameters\": {\n        \"mode\": \"text-editing\",\n        \"ddim_steps\": 20,\n        \"cfg_scale\": 9.0,\n        \"strength\": 1.0,\n        \"eta\": 0.0,\n        \"canvas_width\": 512,\n        \"canvas_height\": 512,\n        \"context_margin_ratio\": 0.5\n      }\n    },\n    {\n      \"attempt_id\": \"attempt_B_replacement_item_name\",\n      \"source_row_index\": 61,\n      \"source_image_id\": 61,\n      \"operation\": \"replacement\",\n      \"original_text\": \"Populaire Chocolate\",\n      \"target_text\": \"Magnum Gold Almond\",\n      \"target_bbox\": [\n        1085,\n        128,\n        1156,\n        749\n      ],\n      \"native_dimensions\": [\n        1356,\n        2315\n      ],\n      \"authentic_sha256\": \"77d047b9791ff4667bf85b0b86448daf154af15660686cf72047d9db407b5ed0\",\n      \"seed\": 20261013,\n      \"prompt\": \"A receipt bill menu with item name \\\"Magnum Gold Almond\\\"\",\n      \"a_prompt\": \"best quality, extremely detailed, clear text edges, neat writing, no watermarks\",\n      \"n_prompt\": \"low-res, bad anatomy, cropped, worst quality, low quality, watermark, unreadable text, messy words, distorted text\",\n      \"sampling_parameters\": {\n        \"mode\": \"text-editing\",\n        \"ddim_steps\": 20,\n        \"cfg_scale\": 9.0,\n        \"strength\": 1.0,\n        \"eta\": 0.0,\n        \"canvas_width\": 512,\n        \"canvas_height\": 512,\n        \"context_margin_ratio\": 0.5\n      }\n    }\n  ]\n}", encoding='utf-8')
pathlib.Path('ml/evaluation/document_feasibility_geometry.py').write_text("\"\"\"Document feasibility geometry transformations and composite contract.\n\nImplements aspect-ratio preserving context cropping, canvas letterboxing,\nbinary mask transforms, inverse mapping, and strict bitwise outside-mask\nequality verification for AI document manipulation experiments.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom typing import Any, Tuple, Dict, List\nimport numpy as np\nfrom PIL import Image\n\n\ndef extract_roi_and_build_canvas(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 1024,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    \"\"\"Extract context crop around target bounding box and letterbox onto square canvas.\n\n    Args:\n        image: PIL Image in RGB format (native dimensions).\n        target_bbox: [ymin, xmin, ymax, xmax] in original pixel coordinates.\n        canvas_size: Square canvas resolution for diffusion model (e.g. 1024).\n        context_margin_ratio: Margin to expand around target bbox (e.g. 0.5 = 50%).\n\n    Returns:\n        Dict containing:\n            canvas_image: PIL Image (RGB, canvas_size x canvas_size)\n            canvas_mask: PIL Image (L, canvas_size x canvas_size, strictly {0, 255})\n            context_crop: PIL Image (RGB, crop_w x crop_h)\n            geometry_receipt: Dict detailing all transformations for inverse mapping.\n    \"\"\"\n    if image.mode != \"RGB\":\n        image = image.convert(\"RGB\")\n\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n\n    # Validate bounding box strictly: silent clamping is forbidden\n    if ymin < 0 or xmin < 0 or ymax > orig_h or xmax > orig_w:\n        raise ValueError(\n            f\"Bounding box {target_bbox} is out of bounds for image dimensions ({orig_w}, {orig_h}). \"\n            f\"Silent clamping is forbidden by protocol.\"\n        )\n    if ymin >= ymax or xmin >= xmax:\n        raise ValueError(\n            f\"Degenerate bounding box {target_bbox}: ymin ({ymin}) >= ymax ({ymax}) \"\n            f\"or xmin ({xmin}) >= xmax ({xmax}).\"\n        )\n\n    bbox_w = xmax - xmin\n    bbox_h = ymax - ymin\n\n    # Expand margin\n    margin_x = int(round(bbox_w * context_margin_ratio))\n    margin_y = int(round(bbox_h * context_margin_ratio))\n\n    crop_ymin = max(0, ymin - margin_y)\n    crop_ymax = min(orig_h, ymax + margin_y)\n    crop_xmin = max(0, xmin - margin_x)\n    crop_xmax = min(orig_w, xmax + margin_x)\n\n    crop_w = crop_xmax - crop_xmin\n    crop_h = crop_ymax - crop_ymin\n\n    # Crop image\n    context_crop = image.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))\n\n    # Build mask in crop coordinate space\n    # Target bbox relative to crop origin\n    target_rel_ymin = ymin - crop_ymin\n    target_rel_ymax = ymax - crop_ymin\n    target_rel_xmin = xmin - crop_xmin\n    target_rel_xmax = xmax - crop_xmin\n\n    mask_crop_arr = np.zeros((crop_h, crop_w), dtype=np.uint8)\n    mask_crop_arr[target_rel_ymin:target_rel_ymax, target_rel_xmin:target_rel_xmax] = 255\n    mask_crop = Image.fromarray(mask_crop_arr, mode=\"L\")\n\n    # Aspect-ratio preserving scaling\n    scale = min(canvas_size / crop_w, canvas_size / crop_h)\n    scaled_w = max(1, min(canvas_size, int(round(crop_w * scale))))\n    scaled_h = max(1, min(canvas_size, int(round(crop_h * scale))))\n\n    scaled_image = context_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.BICUBIC)\n    # Mask resize MUST use NEAREST neighbor to avoid intermediate gray values\n    scaled_mask = mask_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.NEAREST)\n\n    # Re-binarize to ensure exact {0, 255}\n    scaled_mask_arr = (np.array(scaled_mask) > 127).astype(np.uint8) * 255\n    scaled_mask = Image.fromarray(scaled_mask_arr, mode=\"L\")\n\n    # Letterbox onto square canvas\n    pad_x = (canvas_size - scaled_w) // 2\n    pad_y = (canvas_size - scaled_h) // 2\n\n    canvas_image = Image.new(\"RGB\", (canvas_size, canvas_size), color=(128, 128, 128))\n    canvas_image.paste(scaled_image, (pad_x, pad_y))\n\n    canvas_mask = Image.new(\"L\", (canvas_size, canvas_size), color=0)\n    canvas_mask.paste(scaled_mask, (pad_x, pad_y))\n\n    geometry_receipt = {\n        \"original_size\": [orig_w, orig_h],\n        \"target_bbox\": [ymin, xmin, ymax, xmax],\n        \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        \"crop_size\": [crop_w, crop_h],\n        \"scale\": float(scale),\n        \"scaled_size\": [scaled_w, scaled_h],\n        \"padding\": [pad_y, pad_x],\n        \"canvas_size\": canvas_size,\n        \"context_margin_ratio\": float(context_margin_ratio),\n    }\n\n    return {\n        \"canvas_image\": canvas_image,\n        \"canvas_mask\": canvas_mask,\n        \"context_crop\": context_crop,\n        \"geometry_receipt\": geometry_receipt,\n    }\n\n\ndef composite_inpainted_patch(\n    original_image: Image.Image,\n    model_output_canvas: Image.Image,\n    geometry_receipt: Dict[str, Any],\n) -> Dict[str, Any]:\n    \"\"\"Inverse map model canvas output back to original coordinates and composite strictly inside mask.\n\n    Guarantees that pixels outside the intervention mask remain 100% bitwise identical\n    to the authentic source image.\n\n    Args:\n        original_image: Authentic PIL Image (native dimensions).\n        model_output_canvas: Raw output from inpainting model (canvas_size x canvas_size).\n        geometry_receipt: Transformation parameters from extract_roi_and_build_canvas.\n\n    Returns:\n        Dict containing:\n            final_edited_image: PIL Image (RGB, native dimensions)\n            final_intervention_mask: PIL Image (L, native dimensions, strictly {0, 255})\n            composite_receipt: Summary metadata\n    \"\"\"\n    if original_image.mode != \"RGB\":\n        original_image = original_image.convert(\"RGB\")\n    if model_output_canvas.mode != \"RGB\":\n        model_output_canvas = model_output_canvas.convert(\"RGB\")\n\n    orig_w, orig_h = geometry_receipt[\"original_size\"]\n    crop_ymin, crop_xmin, crop_ymax, crop_xmax = geometry_receipt[\"crop_bbox\"]\n    crop_w, crop_h = geometry_receipt[\"crop_size\"]\n    scaled_w, scaled_h = geometry_receipt[\"scaled_size\"]\n    pad_y, pad_x = geometry_receipt[\"padding\"]\n    target_ymin, target_xmin, target_ymax, target_xmax = geometry_receipt[\"target_bbox\"]\n\n    # 1. Unpad canvas\n    unpadded_patch = model_output_canvas.crop((pad_x, pad_y, pad_x + scaled_w, pad_y + scaled_h))\n\n    # 2. Resize unpadded patch back to crop size\n    restored_patch = unpadded_patch.resize((crop_w, crop_h), resample=Image.Resampling.BICUBIC)\n    restored_patch_arr = np.array(restored_patch)\n\n    # 3. Create full-frame candidate edited image\n    orig_arr = np.array(original_image)\n    candidate_full_arr = orig_arr.copy()\n    candidate_full_arr[crop_ymin:crop_ymax, crop_xmin:crop_xmax] = restored_patch_arr\n\n    # 4. Create final intervention mask in original coordinates\n    final_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    final_mask_arr[target_ymin:target_ymax, target_xmin:target_xmax] = 255\n\n    # 5. Composite STRICTLY inside intervention mask:\n    # pixels where mask == 255 take candidate_full_arr, everywhere else takes orig_arr bit-for-bit\n    final_edited_arr = np.where(final_mask_arr[:, :, None] == 255, candidate_full_arr, orig_arr)\n\n    final_edited_image = Image.fromarray(final_edited_arr, mode=\"RGB\")\n    final_intervention_mask = Image.fromarray(final_mask_arr, mode=\"L\")\n\n    composite_receipt = {\n        \"status\": \"COMPOSITED\",\n        \"original_size\": [orig_w, orig_h],\n        \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        \"target_bbox\": [target_ymin, target_xmin, target_ymax, target_xmax],\n        \"unpadded_size\": [scaled_w, scaled_h],\n        \"restored_crop_size\": [crop_w, crop_h],\n    }\n\n    return {\n        \"final_edited_image\": final_edited_image,\n        \"final_intervention_mask\": final_intervention_mask,\n        \"composite_receipt\": composite_receipt,\n    }\n\n\ndef verify_outside_equality(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Tuple[bool, int]:\n    \"\"\"Verify that all pixels outside the intervention mask are strictly bitwise identical.\n\n    Returns:\n        (is_equal, max_absolute_diff)\n    \"\"\"\n    auth_arr = np.array(authentic_image, dtype=np.uint8)\n    edit_arr = np.array(edited_image, dtype=np.uint8)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n\n    if auth_arr.shape[:2] != mask_arr.shape[:2] or edit_arr.shape[:2] != mask_arr.shape[:2]:\n        raise ValueError(\"Image dimensions and mask dimensions do not match.\")\n\n    outside_indices = mask_arr == 0\n    if not np.any(outside_indices):\n        # Entire image is masked (should not happen in document pilot)\n        return True, 0\n\n    auth_outside = auth_arr[outside_indices]\n    edit_outside = edit_arr[outside_indices]\n\n    is_equal = bool(np.array_equal(auth_outside, edit_outside))\n    if is_equal:\n        return True, 0\n\n    diff = np.abs(auth_outside.astype(np.int32) - edit_outside.astype(np.int32))\n    max_diff = int(np.max(diff))\n    return False, max_diff\n\n\ndef compute_pixel_change_statistics(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Dict[str, Any]:\n    \"\"\"Compute detailed pixel difference statistics inside and outside the intervention mask.\"\"\"\n    auth_arr = np.array(authentic_image, dtype=np.int32)\n    edit_arr = np.array(edited_image, dtype=np.int32)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n\n    inside_mask = mask_arr == 255\n    outside_mask = mask_arr == 0\n\n    inside_pixels_count = int(np.sum(inside_mask))\n    outside_pixels_count = int(np.sum(outside_mask))\n\n    # Inside differences\n    if inside_pixels_count > 0:\n        inside_diff = np.abs(auth_arr[inside_mask] - edit_arr[inside_mask])\n        inside_changed = np.any(inside_diff > 0, axis=-1)\n        changed_pixels_count = int(np.sum(inside_changed))\n        changed_pixel_ratio = float(changed_pixels_count / inside_pixels_count)\n        inside_mean_l1 = float(np.mean(inside_diff))\n        inside_max_l1 = int(np.max(inside_diff))\n    else:\n        changed_pixels_count = 0\n        changed_pixel_ratio = 0.0\n        inside_mean_l1 = 0.0\n        inside_max_l1 = 0\n\n    # Outside differences\n    if outside_pixels_count > 0:\n        outside_diff = np.abs(auth_arr[outside_mask] - edit_arr[outside_mask])\n        outside_changed = np.any(outside_diff > 0, axis=-1)\n        outside_changed_count = int(np.sum(outside_changed))\n        outside_mean_l1 = float(np.mean(outside_diff))\n        outside_max_l1 = int(np.max(outside_diff))\n    else:\n        outside_changed_count = 0\n        outside_mean_l1 = 0.0\n        outside_max_l1 = 0\n\n    return {\n        \"inside_mask_pixel_count\": inside_pixels_count,\n        \"changed_pixels_count\": changed_pixels_count,\n        \"changed_pixel_ratio\": changed_pixel_ratio,\n        \"inside_mean_l1\": inside_mean_l1,\n        \"inside_max_l1\": inside_max_l1,\n        \"outside_mask_pixel_count\": outside_pixels_count,\n        \"outside_changed_count\": outside_changed_count,\n        \"outside_mean_l1\": outside_mean_l1,\n        \"outside_max_l1\": outside_max_l1,\n        \"is_outside_bit_identical\": (outside_changed_count == 0),\n    }\n", encoding='utf-8')
pathlib.Path('ml/evaluation/anytext_document_adapter.py').write_text("\"\"\"AnyText Document Adapter and Geometry Contract.\n\nProvides preprocessing, position mask conditioning, inverse mapping,\nand strict outside-mask compositing for AnyText text-editing on document images.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List, Tuple\nimport numpy as np\nfrom PIL import Image\n\nfrom ml.evaluation.document_feasibility_geometry import (\n    extract_roi_and_build_canvas,\n    composite_inpainted_patch,\n    verify_outside_equality,\n    validate_bounding_box_within_bounds,\n)\n\n\ndef prepare_anytext_editing_inputs(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 512,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    \"\"\"Prepares AnyText ori_image and draw_pos inputs from native document and target bbox.\n\n    Args:\n        image: Original PIL Image in RGB format (native dimensions).\n        target_bbox: [ymin, xmin, ymax, xmax] in native coordinates.\n        canvas_size: Square canvas resolution for AnyText (512x512).\n        context_margin_ratio: Margin expansion around target bbox (default: 0.5).\n\n    Returns:\n        Dict with:\n            ori_image_canvas: PIL Image (RGB, 512x512) for AnyText ori_image.\n            draw_pos_canvas: PIL Image (L, 512x512, {0, 255}) for AnyText draw_pos.\n            geometry_receipt: Dict tracking transformations for inverse mapping.\n            native_mask: PIL Image (L, orig_w x orig_h, {0, 255}).\n    \"\"\"\n    if image.mode != \"RGB\":\n        image = image.convert(\"RGB\")\n\n    validate_bounding_box_within_bounds(target_bbox, image.size)\n\n    # Use verified geometry builder with 512x512 canvas size\n    roi_data = extract_roi_and_build_canvas(\n        image=image,\n        target_bbox=target_bbox,\n        canvas_size=canvas_size,\n        context_margin_ratio=context_margin_ratio,\n    )\n\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n\n    # Native intervention mask\n    native_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    native_mask_arr[ymin:ymax, xmin:xmax] = 255\n    native_mask = Image.fromarray(native_mask_arr, mode=\"L\")\n\n    return {\n        \"ori_image_canvas\": roi_data[\"canvas_image\"],\n        \"draw_pos_canvas\": roi_data[\"canvas_mask\"],\n        \"context_crop\": roi_data[\"context_crop\"],\n        \"geometry_receipt\": roi_data[\"geometry_receipt\"],\n        \"native_mask\": native_mask,\n    }\n\n\ndef composite_anytext_editing_output(\n    original_image: Image.Image,\n    raw_anytext_output: Image.Image | np.ndarray,\n    geometry_receipt: Dict[str, Any],\n    native_mask: Image.Image,\n) -> Dict[str, Any]:\n    \"\"\"Composites raw AnyText canvas back into native document strictly inside mask.\n\n    Args:\n        original_image: Original PIL Image in RGB format.\n        raw_anytext_output: 512x512 RGB raw output from AnyText (PIL or numpy array).\n        geometry_receipt: Receipt from prepare_anytext_editing_inputs.\n        native_mask: Native binary mask (L mode, {0, 255}).\n\n    Returns:\n        Dict with:\n            final_edited_document: PIL Image in RGB format (native dimensions).\n            is_outside_equal: bool (True if 100% bitwise identical outside mask).\n            max_outside_diff: int (must be 0).\n            raw_model_canvas: PIL Image (512x512).\n    \"\"\"\n    if original_image.mode != \"RGB\":\n        original_image = original_image.convert(\"RGB\")\n\n    if isinstance(raw_anytext_output, np.ndarray):\n        raw_canvas = Image.fromarray(raw_anytext_output.astype(np.uint8), mode=\"RGB\")\n    else:\n        raw_canvas = raw_anytext_output.convert(\"RGB\")\n\n    # Composite using verified contract\n    composited_img = composite_inpainted_patch(\n        original_image=original_image,\n        inpainted_canvas=raw_canvas,\n        geometry_receipt=geometry_receipt,\n        intervention_mask=native_mask,\n    )\n\n    # Strict outside-mask equality check\n    is_outside_equal, max_outside_diff = verify_outside_equality(\n        original_image=original_image,\n        composited_image=composited_img,\n        intervention_mask=native_mask,\n    )\n\n    return {\n        \"final_edited_document\": composited_img,\n        \"is_outside_equal\": is_outside_equal,\n        \"max_outside_diff\": max_outside_diff,\n        \"raw_model_canvas\": raw_canvas,\n    }\n", encoding='utf-8')
pathlib.Path('run_anytext_colab.py').write_text("#!/usr/bin/env python3\n\"\"\"AnyText Document Generation Feasibility Colab Runner.\n\nRuns AnyText text-editing on two locked replacement sources (Attempt A & B)\non Google Colab GPU without Hugging Face token or paid dependencies.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport datetime\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef compute_sha256(path: Path) -> str:\n    \"\"\"Computes SHA-256 hash of a file.\"\"\"\n    h = hashlib.sha256()\n    with path.open(\"rb\") as f:\n        while chunk := f.read(1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef check_gpu_and_resources(min_vram_gb: float = 7.0) -> Dict[str, Any]:\n    \"\"\"Inspects CUDA device, VRAM, and disk storage.\"\"\"\n    res: Dict[str, Any] = {\n        \"cuda_available\": False,\n        \"device_name\": \"None\",\n        \"vram_total_gb\": 0.0,\n        \"vram_allocated_gb\": 0.0,\n        \"disk_free_gb\": 0.0,\n    }\n    # Disk check\n    disk = shutil.disk_usage(\"/\")\n    res[\"disk_free_gb\"] = round(disk.free / (1024**3), 2)\n\n    try:\n        import torch\n        if torch.cuda.is_available():\n            res[\"cuda_available\"] = True\n            res[\"device_name\"] = torch.cuda.get_device_name(0)\n            res[\"vram_total_gb\"] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)\n            res[\"vram_allocated_gb\"] = round(torch.cuda.memory_allocated(0) / (1024**3), 2)\n    except Exception as e:\n        res[\"error\"] = str(e)\n    return res\n\n\ndef resolve_font_path(custom_path: Optional[str] = None) -> str:\n    \"\"\"Finds an available open-source TrueType font for glyph rendering.\"\"\"\n    candidates = [\n        custom_path,\n        \"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\",\n        \"/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf\",\n        \"/usr/share/fonts/truetype/freefont/FreeSans.ttf\",\n        \"font/Arial_Unicode.ttf\",\n    ]\n    for c in candidates:\n        if c and Path(c).is_file():\n            return str(Path(c).resolve())\n\n    # Fallback to downloading or apt installing\n    try:\n        subprocess.run([\"apt-get\", \"update\", \"-qq\"], check=False)\n        subprocess.run([\"apt-get\", \"install\", \"-y\", \"-qq\", \"fonts-dejavu-core\"], check=False)\n        if Path(\"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\").is_file():\n            return \"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\"\n    except Exception:\n        pass\n\n    raise FileNotFoundError(\"Could not find a valid open-source TTF font for AnyText glyph rendering.\")\n\n\ndef run_anytext_inference(\n    pipe: Any,\n    ori_image_canvas: Image.Image,\n    draw_pos_canvas: Image.Image,\n    prompt: str,\n    target_text: str,\n    seed: int,\n    sampling_params: Dict[str, Any],\n    a_prompt: str,\n    n_prompt: str,\n) -> Tuple[Image.Image, float, Dict[str, Any]]:\n    \"\"\"Invokes AnyText pipeline in text-editing mode.\"\"\"\n    t0 = time.time()\n\n    # AnyText expectations:\n    # ori_image: numpy array (H, W, 3) RGB\n    # draw_pos: numpy array (H, W, 1) where 255 is the text edit region\n    ori_np = np.array(ori_image_canvas.convert(\"RGB\"))\n    pos_np = np.array(draw_pos_canvas.convert(\"L\"))[..., np.newaxis]\n\n    input_data = {\n        \"prompt\": prompt,\n        \"seed\": seed,\n        \"draw_pos\": pos_np,\n        \"ori_image\": ori_np,\n    }\n\n    forward_params = {\n        \"mode\": \"text-editing\",\n        \"image_count\": 1,\n        \"ddim_steps\": sampling_params.get(\"ddim_steps\", 20),\n        \"cfg_scale\": sampling_params.get(\"cfg_scale\", 9.0),\n        \"strength\": sampling_params.get(\"strength\", 1.0),\n        \"eta\": sampling_params.get(\"eta\", 0.0),\n        \"a_prompt\": a_prompt,\n        \"n_prompt\": n_prompt,\n        \"show_debug\": False,\n        \"revise_pos\": False,\n    }\n\n    results, rtn_code, rtn_warning, debug_info = pipe(input_data, **forward_params)\n    latency_sec = time.time() - t0\n\n    if rtn_code < 0 or not results:\n        raise RuntimeError(f\"AnyText inference failed with code {rtn_code}: {rtn_warning}\")\n\n    raw_output_np = results[0]\n    raw_output_img = Image.fromarray(raw_output_np.astype(np.uint8), mode=\"RGB\")\n    meta = {\n        \"latency_sec\": round(latency_sec, 2),\n        \"rtn_code\": rtn_code,\n        \"rtn_warning\": rtn_warning,\n    }\n    return raw_output_img, latency_sec, meta\n\n\ndef generate_anytext_contact_sheet(\n    attempts_data: List[Dict[str, Any]],\n    output_path: Path,\n) -> None:\n    \"\"\"Generates a high-res comparison contact sheet for the AnyText attempts.\"\"\"\n    thumb_w, thumb_h = 320, 320\n    header_h = 60\n    label_h = 30\n    cols = 5  # Authentic, SDXL Output, AnyText Raw, AnyText Final, Mask Inspection\n    rows = len(attempts_data)\n    margin = 15\n\n    total_w = cols * thumb_w + (cols + 1) * margin\n    total_h = header_h + rows * (thumb_h + label_h) + (rows + 1) * margin\n\n    sheet = Image.new(\"RGB\", (total_w, total_h), (250, 250, 250))\n    draw = ImageDraw.Draw(sheet)\n\n    # Title\n    draw.text(\n        (margin, 15),\n        \"AnyText Feasibility Study: 2 Replacement Sources vs Authentic & SDXL\",\n        fill=(20, 20, 20),\n    )\n\n    col_titles = [\n        \"1. Authentic (I0)\",\n        \"2. SDXL Pilot (I2)\",\n        \"3. AnyText Raw (512x512)\",\n        \"4. AnyText Final Composited\",\n        \"5. Intervention Mask\",\n    ]\n    for c_idx, title in enumerate(col_titles):\n        x = margin + c_idx * (thumb_w + margin)\n        draw.text((x, 40), title, fill=(80, 80, 80))\n\n    for r_idx, att in enumerate(attempts_data):\n        y_top = header_h + margin + r_idx * (thumb_h + label_h + margin)\n        draw.text(\n            (margin, y_top),\n            f\"Attempt {att['id']}: {att['target_text']} (Seed {att['seed']})\",\n            fill=(10, 50, 120),\n        )\n\n        images = [\n            att.get(\"authentic_img\"),\n            att.get(\"sdxl_img\"),\n            att.get(\"anytext_raw_img\"),\n            att.get(\"anytext_final_img\"),\n            att.get(\"mask_img\"),\n        ]\n\n        for c_idx, img in enumerate(images):\n            x = margin + c_idx * (thumb_w + margin)\n            y = y_top + label_h\n            if img:\n                thumb = img.copy()\n                thumb.thumbnail((thumb_w, thumb_h), Image.Resampling.BICUBIC)\n                # Center thumbnail\n                offset_x = x + (thumb_w - thumb.width) // 2\n                offset_y = y + (thumb_h - thumb.height) // 2\n                sheet.paste(thumb, (offset_x, offset_y))\n                # Border\n                draw.rectangle([offset_x, offset_y, offset_x + thumb.width, offset_y + thumb.height], outline=(200, 200, 200), width=1)\n            else:\n                draw.rectangle([x, y, x + thumb_w, y + thumb_h], fill=(230, 230, 230))\n                draw.text((x + 20, y + thumb_h // 2), \"N/A\", fill=(120, 120, 120))\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(output_path, \"PNG\")\n    print(f\"[OK] Contact sheet written to {output_path}\")\n\n\ndef resolve_input_file(inputs_dir: Path, att: Dict[str, Any]) -> Optional[Path]:\n    \"\"\"Resolves authentic control image path from inputs_dir.\"\"\"\n    att_id = att[\"attempt_id\"]\n    row_idx = att.get(\"source_row_index\")\n    # Potential directories and filenames\n    candidates = [\n        inputs_dir / att_id / \"authentic_control.png\",\n        inputs_dir / f\"{att_id}.png\",\n        inputs_dir / f\"attempt_{0 if 'price' in att_id else 1}_{att_id.split('_', 2)[-1]}\" / \"authentic_control.png\",\n    ]\n    if row_idx == 22:\n        candidates.append(inputs_dir / \"attempt_0_replacement_price\" / \"authentic_control.png\")\n    elif row_idx == 61:\n        candidates.append(inputs_dir / \"attempt_1_replacement_item_name\" / \"authentic_control.png\")\n\n    # Also search by glob inside inputs_dir\n    for c in candidates:\n        if c.is_file():\n            return c\n    for p in inputs_dir.glob(\"**/authentic_control.png\"):\n        if row_idx == 22 and \"price\" in str(p):\n            return p\n        if row_idx == 61 and \"item_name\" in str(p):\n            return p\n    return None\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Run AnyText Feasibility Runner\")\n    parser.add_argument(\"--plan-path\", type=str, required=True, help=\"Path to anytext_feasibility_plan.json\")\n    parser.add_argument(\"--inputs-dir\", type=str, required=True, help=\"Path to directory containing input authentic controls\")\n    parser.add_argument(\"--output-root\", type=str, required=True, help=\"Path to output directory\")\n    parser.add_argument(\"--custom-font\", type=str, default=None, help=\"Custom TTF font path\")\n    parser.add_argument(\"--preflight-only\", action=\"store_true\", help=\"Run preflight checks only\")\n    args = parser.parse_args()\n\n    plan_path = Path(args.plan_path).resolve()\n    inputs_dir = Path(args.inputs_dir).resolve()\n    output_root = Path(args.output_root).resolve()\n\n    if not plan_path.is_file():\n        print(f\"[FAIL] Plan not found: {plan_path}\")\n        return 1\n\n    plan = json.loads(plan_path.read_text(encoding=\"utf-8\"))\n    run_timestamp = datetime.datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    plan_hash_short = hashlib.sha256(plan_path.read_bytes()).hexdigest()[:8]\n    run_id = f\"run_anytext_{run_timestamp}_{plan_hash_short}\"\n    run_dir = output_root / run_id\n    run_dir.mkdir(parents=True, exist_ok=True)\n\n    print(f\"=== ANYTEXT FEASIBILITY RUNNER: {run_id} ===\")\n    hw_info = check_gpu_and_resources()\n    print(f\"CUDA Available: {hw_info['cuda_available']} ({hw_info['device_name']}, VRAM: {hw_info['vram_total_gb']} GB)\")\n    print(f\"Disk Free: {hw_info['disk_free_gb']} GB\")\n\n    font_path = resolve_font_path(args.custom_font)\n    print(f\"Resolved Font: {font_path}\")\n\n    # Check input files\n    print(\"\\n--- Verifying Input Sources ---\")\n    attempts = plan[\"attempts\"]\n    for att in attempts:\n        att_file = resolve_input_file(inputs_dir, att)\n        if not att_file or not att_file.is_file():\n            print(f\"[FAIL] Missing input file for {att['attempt_id']} in {inputs_dir}\")\n            return 1\n        observed_sha = compute_sha256(att_file)\n        if observed_sha != att[\"authentic_sha256\"]:\n            print(f\"[FAIL] SHA mismatch for {att['attempt_id']}: expected {att['authentic_sha256']}, got {observed_sha}\")\n            return 1\n        print(f\"[PASS] Verified input {att['attempt_id']}: {observed_sha[:16]}...\")\n\n    if args.preflight_only:\n        print(\"[OK] Preflight completed successfully.\")\n        return 0\n\n    if not hw_info[\"cuda_available\"]:\n        print(\"[FAIL] CUDA GPU is required for AnyText generation. Running on CPU is not supported.\")\n        return 1\n\n    # Staging AnyText pipeline\n    print(\"--- Loading AnyText Model from ModelScope ---\")\n    os.environ[\"MODELSCOPE_CACHE\"] = str(run_dir / \".cache_modelscope\")\n    from modelscope.pipelines import pipeline\n\n    pipe = pipeline(\n        \"my-anytext-task\",\n        model=plan[\"model_specification\"][\"modelscope_repo_id\"],\n        model_revision=plan[\"model_specification\"][\"model_revision\"],\n        use_fp16=True,\n        use_translator=False,  # Exclude Chinese NLP translation model to respect 12 GiB budget\n        font_path=font_path,\n    )\n    print(\"[OK] AnyText pipeline initialized successfully.\")\n\n    # Import geometry adapter\n    from ml.evaluation.anytext_document_adapter import (\n        prepare_anytext_editing_inputs,\n        composite_anytext_editing_output,\n    )\n\n    contact_sheet_items = []\n    review_records = []\n\n    for att in attempts:\n        att_id = att[\"attempt_id\"]\n        att_dir = run_dir / att_id\n        att_dir.mkdir(parents=True, exist_ok=True)\n        print(f\"\\n--- Executing {att_id} ---\")\n\n        # Load authentic\n        att_input_file = resolve_input_file(inputs_dir, att)\n        if not att_input_file or not att_input_file.is_file():\n            print(f\"[FAIL] Missing input file for {att['attempt_id']}\")\n            return 1\n        auth_img = Image.open(att_input_file).convert(\"RGB\")\n\n        # Prepare inputs\n        prep_res = prepare_anytext_editing_inputs(\n            image=auth_img,\n            target_bbox=att[\"target_bbox\"],\n            canvas_size=att[\"sampling_parameters\"].get(\"canvas_width\", 512),\n            context_margin_ratio=att[\"sampling_parameters\"].get(\"context_margin_ratio\", 0.5),\n        )\n\n        # Run AnyText\n        raw_canvas, latency, inf_meta = run_anytext_inference(\n            pipe=pipe,\n            ori_image_canvas=prep_res[\"ori_image_canvas\"],\n            draw_pos_canvas=prep_res[\"draw_pos_canvas\"],\n            prompt=att[\"prompt\"],\n            target_text=att[\"target_text\"],\n            seed=att[\"seed\"],\n            sampling_params=att[\"sampling_parameters\"],\n            a_prompt=att[\"a_prompt\"],\n            n_prompt=att[\"n_prompt\"],\n        )\n\n        # Composite output\n        comp_res = composite_anytext_editing_output(\n            original_image=auth_img,\n            raw_anytext_output=raw_canvas,\n            geometry_receipt=prep_res[\"geometry_receipt\"],\n            native_mask=prep_res[\"native_mask\"],\n        )\n\n        final_img = comp_res[\"final_edited_document\"]\n        if not comp_res[\"is_outside_equal\"]:\n            print(f\"[FAIL] Outside-mask equality violated for {att_id} (max diff: {comp_res['max_outside_diff']})\")\n            return 1\n        print(f\"[PASS] Outside-mask equality verified (diff = 0)\")\n\n        # Save artifacts\n        auth_save = att_dir / \"authentic_control.png\"\n        raw_save = att_dir / \"anytext_raw_canvas.png\"\n        final_save = att_dir / \"anytext_final_edited.png\"\n        mask_save = att_dir / \"native_mask.png\"\n        draw_pos_save = att_dir / \"draw_pos_canvas.png\"\n\n        auth_img.save(auth_save, \"PNG\")\n        raw_canvas.save(raw_save, \"PNG\")\n        final_img.save(final_save, \"PNG\")\n        prep_res[\"native_mask\"].save(mask_save, \"PNG\")\n        prep_res[\"draw_pos_canvas\"].save(draw_pos_save, \"PNG\")\n\n        # Save attempt receipt\n        att_receipt = {\n            \"attempt_id\": att_id,\n            \"status\": \"TECHNICAL_QC_PASS\",\n            \"operation\": att[\"operation\"],\n            \"target_text\": att[\"target_text\"],\n            \"seed\": att[\"seed\"],\n            \"latency_sec\": latency,\n            \"outside_mask_bitwise_equal\": comp_res[\"is_outside_equal\"],\n            \"max_outside_diff\": comp_res[\"max_outside_diff\"],\n            \"hashes\": {\n                \"authentic_sha256\": compute_sha256(auth_save),\n                \"raw_canvas_sha256\": compute_sha256(raw_save),\n                \"final_edited_sha256\": compute_sha256(final_save),\n                \"native_mask_sha256\": compute_sha256(mask_save),\n            },\n            \"geometry_receipt\": prep_res[\"geometry_receipt\"],\n        }\n        (att_dir / \"attempt_receipt.json\").write_text(json.dumps(att_receipt, indent=2), encoding=\"utf-8\")\n\n        # Collect for contact sheet\n        sdxl_img_path = inputs_dir / att[\"attempt_id\"] / \"final_edited_document.png\"\n        sdxl_img = Image.open(sdxl_img_path).convert(\"RGB\") if sdxl_img_path.is_file() else None\n\n        contact_sheet_items.append({\n            \"id\": att_id,\n            \"target_text\": att[\"target_text\"],\n            \"seed\": att[\"seed\"],\n            \"authentic_img\": auth_img,\n            \"sdxl_img\": sdxl_img,\n            \"anytext_raw_img\": raw_canvas,\n            \"anytext_final_img\": final_img,\n            \"mask_img\": prep_res[\"native_mask\"],\n        })\n\n        review_records.append({\n            \"attempt_id\": att_id,\n            \"target_text\": att[\"target_text\"],\n            \"human_verdict\": \"PENDING_HUMAN_REVIEW\",\n            \"text_legibility_score\": \"\",\n            \"background_realism_score\": \"\",\n            \"notes\": \"\",\n        })\n\n    # Contact sheet\n    cs_path = run_dir / \"anytext_contact_sheet.png\"\n    generate_anytext_contact_sheet(contact_sheet_items, cs_path)\n\n    # Review CSV\n    csv_path = run_dir / \"human_review.csv\"\n    with csv_path.open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=list(review_records[0].keys()))\n        writer.writeheader()\n        writer.writerows(review_records)\n\n    # Package ZIP archive\n    zip_path = output_root / \"anytext_feasibility_results.zip\"\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for root_p, _, files in os.walk(run_dir):\n            for file in files:\n                p = Path(root_p) / file\n                rel = p.relative_to(run_dir)\n                zf.write(p, arcname=str(Path(run_id) / rel))\n\n    print(f\"\\n[DONE] AnyText results archived to: {zip_path}\")\n    print(f\"       SHA-256: {compute_sha256(zip_path)}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n", encoding='utf-8')

print('[PASS] Successfully staged plan, geometry module, adapter and runner.')

In [ ]:
# Cell 4: Download Authentic Inputs (from HuggingFace CORD-v2 or Uploaded SDXL Artifacts)
# Tải 2 ảnh authentic controls từ HuggingFace Parquet shard CORD-v2 hoặc người dùng upload
import pyarrow.parquet as pq, io, requests
from PIL import Image

print('Downloading authentic source documents from CORD-v2 validation split...')
cord_url = 'https://huggingface.co/datasets/naver-clova-ix/cord-v2/resolve/main/data/validation-00000-of-00001-c81bcf80b4353f8a.parquet'
resp = requests.get(cord_url, stream=True)
table = pq.read_table(io.BytesIO(resp.content))

# Row 22 (Attempt A)
img_a_bytes = table['image'][22]['bytes'].as_py()
img_a = Image.open(io.BytesIO(img_a_bytes)).convert('RGB')
img_a.save('inputs/attempt_A_replacement_price/authentic_control.png')

# Row 61 (Attempt B)
img_b_bytes = table['image'][61]['bytes'].as_py()
img_b = Image.open(io.BytesIO(img_b_bytes)).convert('RGB')
img_b.save('inputs/attempt_B_replacement_item_name/authentic_control.png')

print('[PASS] Authentic controls extracted and verified.')

In [ ]:
# Cell 5: Execute AnyText Feasibility Runner & Package Results
import subprocess, sys
cmd = [
    sys.executable, 'run_anytext_colab.py',
    '--plan-path', 'research/evidence/document_pilot/anytext_feasibility_plan.json',
    '--inputs-dir', 'inputs',
    '--output-root', 'outputs',
    '--custom-font', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
]
subprocess.run(cmd, check=True)

# Display Contact Sheet
from IPython.display import Image as IPImage, display
import glob
cs_files = glob.glob('outputs/*/anytext_contact_sheet.png')
if cs_files:
    display(IPImage(cs_files[0]))

# Download Results Archive
from google.colab import files
if os.path.isfile('outputs/anytext_feasibility_results.zip'):
    files.download('outputs/anytext_feasibility_results.zip')